## Regression 

In [ ]:
import pandas as pd 
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet

from sklearn.metrics import r2_score, root_mean_squared_error

In [ ]:
df = pd.read_csv("housing.csv")
df["rooms_per_household"] = df["total_rooms"] / df["households"]
print("\nMissing per column:\n", df.isna().sum())

In [ ]:
y = df["median_house_value"]
X = df.drop(columns=["median_house_value"])

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    random_state=42
)

numeric_features = ["longitude", "latitude", "housing_median_age", "total_rooms", "total_bedrooms", "population", "households", "median_income", "rooms_per_household"]
categorical_features = ["ocean_proximity"]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


In [ ]:
print("Dtypes:\n", X_train.dtypes)
print("\nMissing per column:\n", X_train.isna().sum())

In [ ]:
X_train.hist(bins=50, figsize=(12,8))
plt.show

Rooms_per_household innehåller outliers. Inte rimligt att fastighet har 100 rum. För att begränsa deras påverkan används clipping. 

In [ ]:
X_train["rooms_per_household"] = X_train["rooms_per_household"].clip(upper=30)

In [ ]:
sns.boxplot(x=X["ocean_proximity"], y=y)
plt.xticks(rotation=45)
plt.title("Boxplot for ocean proximity")
plt.tight_layout()
plt.savefig("box_plot_ocean_proximity.png", dpi=150)
plt.show()

In [ ]:
df_for_matrix = X_train.copy()
df_for_matrix["median_house_value"] = y_train

correlation_matrix = df_for_matrix.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(10,5))
sns.heatmap(correlation_matrix, annot=True, ax=ax)
plt.tight_layout()

plt.savefig("correlation_matrix.png", dpi=150)
plt.show()

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

In [ ]:
baseline_linear_pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LinearRegression())
])

ridge_pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", Ridge())
])

lasso_pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", Lasso(max_iter=10000))
])

elastic_pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", ElasticNet(max_iter=10000))
])

models = {
    "Linear Regression": baseline_linear_pipe,
    "Ridge": ridge_pipe,
    "Lasso": lasso_pipe,
    "ElasticNet": elastic_pipe
}

In [ ]:
results = []

for name, model in models.items():
    rmse_scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="neg_root_mean_squared_error"
    )

    r2_scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="r2"
    )

    results.append({
        "model": name,
        "RMSE": -rmse_scores.mean(),
        "R2": r2_scores.mean()
    })

results_df = pd.DataFrame(results)
results_df

Ridge är den modellen som har lägst RMSE, alltså minst fel vid prediktion. Ridge är då den modellen som väljs för att gå vidare med. 

In [ ]:
ridge_param_grid = {
    "model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]
}

ridge_grid = GridSearchCV(
        estimator=ridge_pipe,
        param_grid=ridge_param_grid,
        scoring="neg_root_mean_squared_error",
        cv=5,
        n_jobs=-1
    )

ridge_grid.fit(X_train, y_train)


print("Best RMSE:", -ridge_grid.best_score_)
print("Best parameters", ridge_grid.best_params_)

Ridge optimeras mot metric RMSE. Den parametern som väljs att optimera är alpha. Alpha styr hur mycket modellen straffar stora koefficienter. 

In [ ]:
best_model = ridge_grid.best_estimator_

y_pred = best_model.predict(X_test)

rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

result_best_model_df = pd.DataFrame({
    "Model": ["Ridge - train", "Ridge - test"],
    "RMSE": [round(results_df.iloc[0,1], 0), round(rmse, 0)],
    "R²": [round(results_df.iloc[1,2], 2), round(r2, 2)]
})

result_best_model_df

## KMeans

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X_train_numeric = X_train.drop(columns=["ocean_proximity"])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train_numeric)

inertias = []

K_range = range(2, 11)

for K in K_range:
    km = KMeans(n_clusters=K, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)


plt.figure()
plt.plot(list(K_range), inertias, marker="o")
plt.xlabel("K")
plt.ylabel("Inertia")
plt.title("Elbow: Inertia vs K (X_scaled)")
plt.grid(True)
plt.show()

In [ ]:
results = []

for K in range(2, 11):
    km = KMeans(n_clusters=K, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels)
    results.append({ "K": K, "inertia": km.inertia_, "silhouette": sil })

results_sil_df = pd.DataFrame(results)
display(results_sil_df)

plt.figure()
plt.plot(results_sil_df["K"], results_sil_df["silhouette"], marker="o")
plt.xlabel("K")
plt.ylabel("silhouette")
plt.title("Silhouette vs K (X_scaled)")
plt.grid(True)
plt.show()

In [ ]:
kmeans_2 = KMeans(n_clusters=2, random_state=42, n_init=10)
print("Value count for 2 clusters:")
print(pd.Series(kmeans_2.fit_predict(X_scaled)).value_counts().sort_index())

kmeans_3 = KMeans(n_clusters=3, random_state=42, n_init=10)
print("\nValue count for 3 clusters:")
print(pd.Series(kmeans_3.fit_predict(X_scaled)).value_counts().sort_index())

In [ ]:
cluster_labels = kmeans_3.fit_predict(X_scaled)
X_with_clusters = X_train_numeric.copy()

X_with_clusters["cluster"] = cluster_labels
profile_orig = X_with_clusters.groupby("cluster").mean()
display(profile_orig)